## Datalab Semester 2, Sprint 3

In [1]:
import os
import sqlite3
import pandas as pd

# 1. Zoek de map op waar dit notebook-bestand staat
map_van_notebook = os.path.dirname(os.path.abspath('__file__'))

# 2. Maak het volledige pad naar de database
db_pad = os.path.join(map_van_notebook, 'database.sqlite')
conn = sqlite3.connect(db_pad)

1A


Bepaal met behulp van SQL in deze sprint het volgende:

1A Toon het aantal wedstrijden dat jouw team heeft gespeeld per seizoen.

In [2]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tabellen = pd.read_sql_query(query, conn)

print(tabellen)
query_zoek_team = "SELECT team_long_name FROM Team WHERE team_long_name LIKE '%Barcelona%';"
barca_naam = pd.read_sql_query(query_zoek_team, conn)

print(barca_naam)

                name
0    sqlite_sequence
1  Player_Attributes
2             Player
3              Match
4             League
5            Country
6               Team
7    Team_Attributes
  team_long_name
0   FC Barcelona


In [76]:
query = """
SELECT 
    season, 
    COUNT(*) AS aantal_wedstrijden
FROM 
    Match
WHERE 
    home_team_api_id = (SELECT team_api_id FROM Team WHERE team_long_name = 'FC Barcelona')
    OR 
    away_team_api_id = (SELECT team_api_id FROM Team WHERE team_long_name = 'FC Barcelona')
GROUP BY 
    season;
"""

df_wedstrijden = pd.read_sql_query(query, conn)
df_wedstrijden

,season,aantal_wedstrijden
0,2008/2009,38
1,2009/2010,38
2,2010/2011,38
3,2011/2012,38
4,2012/2013,38
5,2013/2014,38
6,2014/2015,38
7,2015/2016,38


1B

In [77]:
query_year = """
SELECT 
    season,
    COUNT(*) AS aantal_wedstrijden
FROM Match m
JOIN Team t1 ON m.home_team_api_id = t1.team_api_id
JOIN Team t2 ON m.away_team_api_id = t2.team_api_id
WHERE 
    (t1.team_long_name = 'FC Barcelona'
     OR t2.team_long_name = 'FC Barcelona')
    AND strftime('%Y', m.date) = '2010'
GROUP BY season
ORDER BY season;

"""
df_wedstrijden_2010 = pd.read_sql_query(query_year, conn)
df_wedstrijden_2010

,season,aantal_wedstrijden
0,2009/2010,23
1,2010/2011,16


1C


In [78]:
def bepaal_match_punten(row):
    """
    Berekent de punten voor de thuis- en uitploeg op basis van de wedstrijdscore.
    
    Args:
        row (pd.Series): Een rij uit de Match dataframe met 'home_team_goal' en 'away_team_goal'.
        
    Returns:
        pd.Series: De behaalde punten voor [home_points, away_points].
    """
    if row['home_team_goal'] > row['away_team_goal']:
        return pd.Series([3, 0], index=['home_points', 'away_points'])
    elif row['home_team_goal'] < row['away_team_goal']:
        return pd.Series([0, 3], index=['home_points', 'away_points'])
    else:
        return pd.Series([1, 1], index=['home_points', 'away_points'])


def bereken_punten_alle_seizoenen(league_id, connection):
    """
    Haalt wedstrijddata op voor een hele competitie en berekent de punten en het doelsaldo per team, per seizoen.
    
    Args:
        league_id (int): De ID van de gekozen competitie.
        connection (sqlite3.Connection): De database verbinding.
        
    Returns:
        pd.DataFrame: Een dataframe met seizoenen, teamnamen, totale punten en doelsaldo.
    """
    # 1. Haal alle wedstrijden op van de competitie
    query = f"SELECT season, home_team_api_id, away_team_api_id, home_team_goal, away_team_goal FROM Match WHERE league_id = {league_id}"
    df_matches = pd.read_sql_query(query, connection)
    
    # 2. De gekopieerde hulpfunctie om de punten per wedstrijd te berekenen
    df_matches[['home_points', 'away_points']] = df_matches.apply(bepaal_match_punten, axis=1)
    
    # 3. De thuis en uitpunten, maar nu PER SEIZOEN en PER TEAM (en inclusief doelpunten!)
    home_stats = df_matches.groupby(['season', 'home_team_api_id'])[['home_points', 'home_team_goal', 'away_team_goal']].sum().reset_index()
    away_stats = df_matches.groupby(['season', 'away_team_api_id'])[['away_points', 'away_team_goal', 'home_team_goal']].sum().reset_index()
    
    home_stats.columns = ['season', 'team_api_id', 'points', 'doelpunten_voor', 'doelpunten_tegen']
    away_stats.columns = ['season', 'team_api_id', 'points', 'doelpunten_voor', 'doelpunten_tegen']
    
    # 4. Plakken thuis en uit onder elkaar.
    alle_stats = pd.concat([home_stats, away_stats])
    # Telt nu punten, doelpunten_voor en doelpunten_tegen bij elkaar op
    ranglijst_seizoenen = alle_stats.groupby(['season', 'team_api_id']).sum().reset_index()
    
    # Doelsaldo berekenen
    ranglijst_seizoenen['doelsaldo'] = ranglijst_seizoenen['doelpunten_voor'] - ranglijst_seizoenen['doelpunten_tegen']
    
    # 5. Clubnamen en de Team tabel
    df_teams_names = pd.read_sql_query("SELECT team_api_id, team_long_name FROM Team", connection)
    ranglijst_seizoenen = ranglijst_seizoenen.merge(df_teams_names, on='team_api_id')
    
    # Eerst op seizoen, dan op punten, en bij gelijke punten op doelsaldo
    eind_ranglijst = ranglijst_seizoenen.sort_values(by=['season', 'points', 'doelsaldo'], ascending=[True, False, False]).reset_index(drop=True)
    
    eind_ranglijst.index = eind_ranglijst.index + 1
    
    return eind_ranglijst[['season', 'team_long_name', 'points', 'doelsaldo']]

In [79]:
# hele League tabel om de ID's te bekijken
query_competities = "SELECT * FROM League"
df_competities = pd.read_sql_query(query_competities, conn)

display(df_competities)

,id,country_id,name
0,1,1,Belgium Jupiler League
1,1729,1729,England Premier League
2,4769,4769,France Ligue 1
3,7809,7809,Germany 1. Bundesliga
4,10257,10257,Italy Serie A
5,13274,13274,Netherlands Eredivisie
6,15722,15722,Poland Ekstraklasa
7,17642,17642,Portugal Liga ZON Sagres
8,19694,19694,Scotland Premier League
9,21518,21518,Spain LIGA BBVA


In [80]:

# ID  van de Spain LIGA BBVA
mijn_competitie_id = 21518 

# Zet de rekenmachine aan voor alle seizoenen
df_opdracht_1c = bereken_punten_alle_seizoenen(mijn_competitie_id, conn)

# reslutaten printen
display(df_opdracht_1c.head(20))


,season,team_long_name,points,doelsaldo
1,2008/2009,FC Barcelona,87,70
2,2008/2009,Real Madrid CF,78,31
3,2008/2009,Sevilla FC,70,15
4,2008/2009,Atlético Madrid,67,23
5,2008/2009,Villarreal CF,65,7
6,2008/2009,Valencia CF,62,14
7,2008/2009,RC Deportivo de La Coruña,58,1
8,2008/2009,Málaga CF,55,-4
9,2008/2009,RCD Mallorca,51,-7
10,2008/2009,RCD Espanyol,47,-3


1D




Ons team is geiendigt op de 1ste plaatst binnen de competitie wat in de ranglijst is te zien. 